Imports & DB connect

In [2]:
# Cell 1: Imports & DuckDB connection
import os
import numpy as np
import pandas as pd
import duckdb
from IPython.display import display

# Connect to your DuckDB (adjust path if needed)
DB_PATH = "../data/mydb2024-25.duckdb"
con = duckdb.connect(database=DB_PATH, read_only=False)

print("✓ Connected to DuckDB:", DB_PATH)


✓ Connected to DuckDB: ../data/mydb2024-25.duckdb


In [3]:
# Cell 2: Load base tables (only what's needed)
# We enrich the *already-built* 'kinexon_events_detected_enriched' table.

# 1) Enriched detected events
df_enriched = con.execute("""
    SELECT *
    FROM kinexon_events_detected_enriched
""").fetchdf()

print(f"Loaded kinexon_events_detected_enriched: {len(df_enriched):,} rows")
display(df_enriched.head(3))

# 2) Human-entered match_events (restrict to goals for precision)
df_me = con.execute("""
    SELECT fixtureId, eventId, eventTime, eventType, personId, goalKeeperId, playId
    FROM match_events
    WHERE eventType = 'goal'
""").fetchdf()

print(f"Loaded match_events (goals only): {len(df_me):,} rows")
display(df_me.head(3))


Loaded kinexon_events_detected_enriched: 1,296 rows


,timestamp,timestamp_ms,timezone_id,game_clock,period,player_id,distance,speed_ball,trajectory,shot_position_x,...,acceleration in m/s2,total distance in m,metabolic power in W/kg,acceleration load,Unnamed: 17,session_id_pos,fixture_id_pos,match_time_diff_ms,match_eventId,match_eventType
0,2024-10-20 13:05:15,1729429515887,385,0:47,,1480,7.710908,17.267044,"-17.06,-7.13;-17.06,-7.13",-17.055626,...,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,None,None
1,2024-10-20 13:05:51,1729429551049,385,1:23,,970,5.852670,23.651764,"-14.16,-0.37;-14.16,-0.37",-14.158737,...,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,3111.0,09e0f481-8ee4-11ef-baff-6d233f85da92,goal
2,2024-10-20 13:06:27,1729429587462,385,1:59,,1480,6.557843,26.578629,"-15.11,-4.37;-15.11,-4.37",-15.113567,...,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,25894.0,2d298c41-8ee4-11ef-baff-6d233f85da92,goal


Loaded match_events (goals only): 28,539 rows


,fixtureId,eventId,eventTime,eventType,personId,goalKeeperId,playId
0,00c08679-4374-11ef-80bd-73cf0bc66b45,09e0f481-8ee4-11ef-baff-6d233f85da92,2024-10-20T13:05:54.160Z,goal,f24b6a05-3954-11ef-a821-4143a9146400,454487a1-3953-11ef-b79a-855e48b08f9c,09e0f480-8ee4-11ef-baff-6d233f85da92
1,00c08679-4374-11ef-80bd-73cf0bc66b45,2d298c41-8ee4-11ef-baff-6d233f85da92,2024-10-20T13:06:53.356Z,goal,4471a641-3953-11ef-8b14-855e48b08f9c,454487a1-3953-11ef-b79a-855e48b08f9c,2d298c40-8ee4-11ef-baff-6d233f85da92
2,00c08679-4374-11ef-80bd-73cf0bc66b45,45066771-8ee4-11ef-baff-6d233f85da92,2024-10-20T13:07:33.431Z,goal,f2fc1b0a-3954-11ef-ae4a-4143a9146400,454487a1-3953-11ef-b79a-855e48b08f9c,45066770-8ee4-11ef-baff-6d233f85da92


In [4]:
# Cell 3: Normalize timestamps to milliseconds (UTC)

# Ensure enriched has timestamp_ms and fixture_id
assert "timestamp_ms" in df_enriched.columns, "timestamp_ms missing in kinexon_events_detected_enriched"
assert "fixture_id"  in df_enriched.columns,  "fixture_id missing in kinexon_events_detected_enriched"

# Parse match_events eventTime → ms
df_me["eventTime_ms"] = (
    pd.to_datetime(df_me["eventTime"], utc=True, errors="coerce")
      .astype("int64") // 10**6
)

# Coerce enriched timestamps to numeric
df_enriched["timestamp_ms"] = pd.to_numeric(df_enriched["timestamp_ms"], errors="coerce")

# Basic sanity
print("Time normalization summary:")
print("  enriched timestamp_ms nulls:", df_enriched["timestamp_ms"].isna().sum())
print("  match_events eventTime_ms nulls:", df_me["eventTime_ms"].isna().sum())


Time normalization summary:
  enriched timestamp_ms nulls: 0
  match_events eventTime_ms nulls: 0


In [5]:
# Cell 4: Vectorized nearest-neighbor on absolute time within each fixture_id
TOLERANCE_MS = 30_000  # ±30 seconds

def attach_event_ids_by_time(df_ev: pd.DataFrame, df_goals: pd.DataFrame, tol_ms: int) -> pd.DataFrame:
    """
    For each fixture_id:
      - sort goal times
      - for each enriched row, pick nearest goal time by absolute delta
      - keep only if |Δt| <= tol_ms
    Adds: match_eventId, match_eventType, match_time_diff_ms
    """
    df_ev = df_ev.copy()
    df_ev["fixture_id"] = df_ev["fixture_id"].astype(str)
    df_goals = df_goals.copy()
    df_goals["fixtureId"] = df_goals["fixtureId"].astype(str)

    out = []

    for fixture_id, ev_grp in df_ev.groupby("fixture_id", dropna=False):
        me_grp = df_goals[df_goals["fixtureId"] == fixture_id]
        if me_grp.empty:
            # No goals for this fixture → fill NA columns
            ev_grp = ev_grp.assign(
                match_eventId=pd.NA,
                match_eventType=pd.NA,
                match_time_diff_ms=pd.NA,
            )
            out.append(ev_grp)
            continue

        me_grp = me_grp.sort_values("eventTime_ms").reset_index(drop=True)
        me_times = me_grp["eventTime_ms"].to_numpy(dtype="int64")

        ev_times = ev_grp["timestamp_ms"].to_numpy(dtype="int64")
        idx_right = np.searchsorted(me_times, ev_times, side="left")
        idx_left = (idx_right - 1).clip(min=0)

        best_idx = []
        best_delta = []
        for t, il, ir in zip(ev_times, idx_left, idx_right):
            cand = []
            if 0 <= il < len(me_times):
                cand.append((il, me_times[il] - t))
            if 0 <= ir < len(me_times):
                cand.append((ir, me_times[ir] - t))
            if not cand:
                best_idx.append(None)
                best_delta.append(None)
                continue

            pick = min(cand, key=lambda x: abs(x[1]))
            if abs(pick[1]) <= tol_ms:
                best_idx.append(pick[0])
                best_delta.append(int(pick[1]))
            else:
                best_idx.append(None)
                best_delta.append(None)

        ev_out = ev_grp.reset_index(drop=True).copy()
        ev_out["__best_idx"] = best_idx
        ev_out["match_time_diff_ms"] = best_delta
        ev_out["match_eventId"] = pd.NA
        ev_out["match_eventType"] = pd.NA

        mask = ev_out["__best_idx"].notna()
        if mask.any():
            picked = me_grp.iloc[ev_out.loc[mask, "__best_idx"].astype(int)].reset_index(drop=True)
            ev_out.loc[mask, "match_eventId"] = picked["eventId"].values
            ev_out.loc[mask, "match_eventType"] = picked["eventType"].values

        out.append(ev_out.drop(columns="__best_idx"))

    return pd.concat(out, ignore_index=True)

df_enriched_matched = attach_event_ids_by_time(df_enriched, df_me, TOLERANCE_MS)

# Quick summary
matched = df_enriched_matched["match_eventId"].notna().sum()
total   = len(df_enriched_matched)
print(f"Matched events: {matched:,} / {total:,} ({matched/total:.1%}) within ±{TOLERANCE_MS/1000:.0f}s")
display(df_enriched_matched[["fixture_id","timestamp_ms","match_eventId","match_eventType","match_time_diff_ms"]].head(10))


Matched events: 1,189 / 1,296 (91.7%) within ±30s


,fixture_id,timestamp_ms,match_eventId,match_eventType,match_time_diff_ms
0,00c08679-4374-11ef-80bd-73cf0bc66b45,1729429515887,<NA>,<NA>,NaN
1,00c08679-4374-11ef-80bd-73cf0bc66b45,1729429551049,09e0f481-8ee4-11ef-baff-6d233f85da92,goal,3111.0
2,00c08679-4374-11ef-80bd-73cf0bc66b45,1729429587462,2d298c41-8ee4-11ef-baff-6d233f85da92,goal,25894.0
3,00c08679-4374-11ef-80bd-73cf0bc66b45,1729429646928,45066771-8ee4-11ef-baff-6d233f85da92,goal,6503.0
4,00c08679-4374-11ef-80bd-73cf0bc66b45,1729429651504,45066771-8ee4-11ef-baff-6d233f85da92,goal,1927.0
5,00c08679-4374-11ef-80bd-73cf0bc66b45,1729429677454,5444d6e1-8ee4-11ef-baff-6d233f85da92,goal,1512.0
6,00c08679-4374-11ef-80bd-73cf0bc66b45,1729429709510,67cc1391-8ee4-11ef-baff-6d233f85da92,goal,2219.0
7,00c08679-4374-11ef-80bd-73cf0bc66b45,1729429717258,6cf6b550-8ee4-11ef-baff-6d233f85da92,goal,3139.0
8,00c08679-4374-11ef-80bd-73cf0bc66b45,1729429727811,733d2750-8ee4-11ef-baff-6d233f85da92,goal,3113.0
9,00c08679-4374-11ef-80bd-73cf0bc66b45,1729429784611,<NA>,<NA>,NaN


In [6]:
# Cell 5: Inspect time deltas and a few concrete matches

# Distribution of absolute deltas (ms)
delta = df_enriched_matched["match_time_diff_ms"].dropna().astype(int).abs()
if not delta.empty:
    print("Δt (ms) — basic stats within tolerance:")
    print(delta.describe().to_string())
else:
    print("No matches found within tolerance. Consider increasing TOLERANCE_MS.")

# Show a few positive samples
sample_cols = [
    "fixture_id", "timestamp", "timestamp_ms",
    "event_type", "player_id", "success",
    "match_eventId", "match_eventType", "match_time_diff_ms"
]
display(df_enriched_matched[df_enriched_matched["match_eventId"].notna()][sample_cols].head(15))


Δt (ms) — basic stats within tolerance:
count     1189.000000
mean      6122.253154
std       6414.785913
min         26.000000
25%       2343.000000
50%       3420.000000
75%       6783.000000
max      30000.000000


,fixture_id,timestamp,timestamp_ms,event_type,player_id,success,match_eventId,match_eventType,match_time_diff_ms
1,00c08679-4374-11ef-80bd-73cf0bc66b45,2024-10-20 13:05:51,1729429551049,detected_shot_handball,970,1,09e0f481-8ee4-11ef-baff-6d233f85da92,goal,3111.0
2,00c08679-4374-11ef-80bd-73cf0bc66b45,2024-10-20 13:06:27,1729429587462,detected_shot_handball,1480,1,2d298c41-8ee4-11ef-baff-6d233f85da92,goal,25894.0
3,00c08679-4374-11ef-80bd-73cf0bc66b45,2024-10-20 13:07:26,1729429646928,detected_shot_handball,1481,0,45066771-8ee4-11ef-baff-6d233f85da92,goal,6503.0
4,00c08679-4374-11ef-80bd-73cf0bc66b45,2024-10-20 13:07:31,1729429651504,detected_shot_handball,1106,0,45066771-8ee4-11ef-baff-6d233f85da92,goal,1927.0
5,00c08679-4374-11ef-80bd-73cf0bc66b45,2024-10-20 13:07:57,1729429677454,detected_shot_handball,2024,1,5444d6e1-8ee4-11ef-baff-6d233f85da92,goal,1512.0
6,00c08679-4374-11ef-80bd-73cf0bc66b45,2024-10-20 13:08:29,1729429709510,detected_shot_handball,1833,1,67cc1391-8ee4-11ef-baff-6d233f85da92,goal,2219.0
7,00c08679-4374-11ef-80bd-73cf0bc66b45,2024-10-20 13:08:37,1729429717258,detected_shot_handball,1480,1,6cf6b550-8ee4-11ef-baff-6d233f85da92,goal,3139.0
8,00c08679-4374-11ef-80bd-73cf0bc66b45,2024-10-20 13:08:47,1729429727811,detected_shot_handball,1522,1,733d2750-8ee4-11ef-baff-6d233f85da92,goal,3113.0
10,00c08679-4374-11ef-80bd-73cf0bc66b45,2024-10-20 13:10:25,1729429825295,detected_shot_handball,1160,0,bd5e8271-8ee4-11ef-914a-5f491cb312bd,goal,30000.0
11,00c08679-4374-11ef-80bd-73cf0bc66b45,2024-10-20 13:11:55,1729429915114,detected_shot_handball,1160,0,e21e50e1-8ee4-11ef-914a-5f491cb312bd,goal,1836.0


In [7]:
# Cell 6: Persist result (overwrite existing table with the new columns)

TABLE_NAME = "kinexon_events_detected_enriched"

print(f"Overwriting table: {TABLE_NAME}")
con.execute(f"DROP TABLE IF EXISTS {TABLE_NAME}")
con.register("df", df_enriched_matched)
con.execute(f"CREATE TABLE {TABLE_NAME} AS SELECT * FROM df")
con.unregister("df")

# Verify write
rows = con.execute(f"SELECT COUNT(*) FROM {TABLE_NAME}").fetchone()[0]
print(f"✓ Wrote {rows:,} rows to {TABLE_NAME}")

# Quick spot-check from DB
check = con.execute(f"""
    SELECT fixture_id, timestamp_ms, match_eventId, match_eventType, match_time_diff_ms
    FROM {TABLE_NAME}
    WHERE match_eventId IS NOT NULL
    LIMIT 10
""").fetchdf()
display(check)


Overwriting table: kinexon_events_detected_enriched
✓ Wrote 1,296 rows to kinexon_events_detected_enriched


,fixture_id,timestamp_ms,match_eventId,match_eventType,match_time_diff_ms
0,00c08679-4374-11ef-80bd-73cf0bc66b45,1729429551049,09e0f481-8ee4-11ef-baff-6d233f85da92,goal,3111.0
1,00c08679-4374-11ef-80bd-73cf0bc66b45,1729429587462,2d298c41-8ee4-11ef-baff-6d233f85da92,goal,25894.0
2,00c08679-4374-11ef-80bd-73cf0bc66b45,1729429646928,45066771-8ee4-11ef-baff-6d233f85da92,goal,6503.0
3,00c08679-4374-11ef-80bd-73cf0bc66b45,1729429651504,45066771-8ee4-11ef-baff-6d233f85da92,goal,1927.0
4,00c08679-4374-11ef-80bd-73cf0bc66b45,1729429677454,5444d6e1-8ee4-11ef-baff-6d233f85da92,goal,1512.0
5,00c08679-4374-11ef-80bd-73cf0bc66b45,1729429709510,67cc1391-8ee4-11ef-baff-6d233f85da92,goal,2219.0
6,00c08679-4374-11ef-80bd-73cf0bc66b45,1729429717258,6cf6b550-8ee4-11ef-baff-6d233f85da92,goal,3139.0
7,00c08679-4374-11ef-80bd-73cf0bc66b45,1729429727811,733d2750-8ee4-11ef-baff-6d233f85da92,goal,3113.0
8,00c08679-4374-11ef-80bd-73cf0bc66b45,1729429825295,bd5e8271-8ee4-11ef-914a-5f491cb312bd,goal,30000.0
9,00c08679-4374-11ef-80bd-73cf0bc66b45,1729429915114,e21e50e1-8ee4-11ef-914a-5f491cb312bd,goal,1836.0


In [8]:
# Cell 7 (optional): Create a helper view focusing on match linkage

VIEW_NAME = "vw_kinexon_events_detected_with_matchid"
con.execute(f"DROP VIEW IF EXISTS {VIEW_NAME}")
con.execute(f"""
    CREATE VIEW {VIEW_NAME} AS
    SELECT
        fixture_id,
        session_id,
        id AS kinexon_event_id,
        event_type,
        player_id,
        timestamp,
        timestamp_ms,
        success,
        shot_category,
        shot_position_x,
        shot_position_y,
        match_eventId,
        match_eventType,
        match_time_diff_ms
    FROM kinexon_events_detected_enriched
""")
print(f"✓ Created view: {VIEW_NAME}")

display(con.execute(f"SELECT * FROM {VIEW_NAME}").fetchdf())


✓ Created view: vw_kinexon_events_detected_with_matchid


,fixture_id,session_id,kinexon_event_id,event_type,player_id,timestamp,timestamp_ms,success,shot_category,shot_position_x,shot_position_y,match_eventId,match_eventType,match_time_diff_ms
0,00c08679-4374-11ef-80bd-73cf0bc66b45,2658,15328703,detected_shot_handball,1480,2024-10-20 13:05:15,1729429515887,0,field,-17.055626,-7.126623,None,None,NaN
1,00c08679-4374-11ef-80bd-73cf0bc66b45,2658,15328789,detected_shot_handball,970,2024-10-20 13:05:51,1729429551049,1,field,-14.158737,-0.365233,09e0f481-8ee4-11ef-baff-6d233f85da92,goal,3111.0
2,00c08679-4374-11ef-80bd-73cf0bc66b45,2658,15328957,detected_shot_handball,1480,2024-10-20 13:06:27,1729429587462,1,field,-15.113567,-4.373566,2d298c41-8ee4-11ef-baff-6d233f85da92,goal,25894.0
3,00c08679-4374-11ef-80bd-73cf0bc66b45,2658,15329202,detected_shot_handball,1481,2024-10-20 13:07:26,1729429646928,0,field,-11.447849,-3.409638,45066771-8ee4-11ef-baff-6d233f85da92,goal,6503.0
4,00c08679-4374-11ef-80bd-73cf0bc66b45,2658,15329278,detected_shot_handball,1106,2024-10-20 13:07:31,1729429651504,0,field,7.307405,-7.454543,45066771-8ee4-11ef-baff-6d233f85da92,goal,1927.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1291,09ff9ff5-4374-11ef-a150-7f7ffdaf25a2,2675,16549282,detected_shot_handball,2079,2024-10-31 19:18:08,1730402288000,0,field,NaN,NaN,d846f731-97bc-11ef-9ea7-33f021d29317,goal,-6965.0
1292,09ff9ff5-4374-11ef-a150-7f7ffdaf25a2,2675,16549283,detected_shot_handball,1598,2024-10-31 19:19:44,1730402384000,0,field,NaN,NaN,1c5859a1-97bd-11ef-9ea7-33f021d29317,goal,1959.0
1293,09ff9ff5-4374-11ef-a150-7f7ffdaf25a2,2675,16763398,detected_shot_handball,1957,2024-10-31 18:57:49,1730401069000,1,field,NaN,NaN,052e8221-97ba-11ef-9ea7-33f021d29317,goal,-1118.0
1294,09ff9ff5-4374-11ef-a150-7f7ffdaf25a2,2675,16763399,detected_shot_handball,1598,2024-10-31 19:18:50,1730402330000,1,field,NaN,NaN,f7f85e70-97bc-11ef-9ea7-33f021d29317,goal,4206.0


In [ ]:
# Cell 8: Prepare positions, helper lookups, and styling

import os
from pathlib import Path
import numpy as np
import pandas as pd
import cv2
from IPython.display import display

# --- parameters ---
POS_PAD_SEC_BEFORE = 15   # seconds before event time
POS_PAD_SEC_AFTER  = 0    # seconds after event time (keep 0 for goal-at-end)
FPS = 60
SHOW_WINDOW = True  # auto-disable in headless

# Output
OUT_DIR = Path("data/metadata")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Field image
FIELD_IMAGE = Path("data/metadata/handballfeld.png")
if not FIELD_IMAGE.exists():
    alt = Path("..") / FIELD_IMAGE
    FIELD_IMAGE = alt if alt.exists() else Path("data/metadata/handballfeld.png")

# Load positions once (for the current notebook context; restrict by session_ids used earlier if you want)
# If you render fixture-by-fixture, you can narrow to those session_ids to speed up
df_positions_all = con.execute("""
    SELECT *
    FROM kinexon_positions
""").fetchdf()

# Pre-compute a UTC datetime column for all positions (ms → datetime)
if "ts" not in df_positions_all.columns:
    df_positions_all["ts"] = pd.to_datetime(df_positions_all["ts in ms"], unit="ms", utc=True, errors="coerce")

# Useful color/style helpers
def color_for_group(group_id: float):
    # 3 = ball, 2 = away team?, 1 = home team? (your feed may differ, keep your mapping)
    # Use bright, distinct colors
    if group_id == 3:
        return (0, 0, 255), 15   # red-ish for ball
    if group_id == 2:
        return (0, 200, 0), 10   # green for one team
    if group_id == 1:
        return (255, 140, 0), 10 # blue/orange-ish for other team
    return (200, 200, 200), 8

def has_display():
    # Basic check to avoid cv2.imshow on headless machines
    return SHOW_WINDOW

print(f"Positions loaded: {len(df_positions_all):,} rows")
display(df_positions_all.head(3))


Positions loaded: 15,618,434 rows


,ts in ms,formatted local time,sensor id,mapped id,number,full name,league id,group id,group name,x in m,...,speed in m/s,direction of movement in deg,acceleration in m/s2,total distance in m,metabolic power in W/kg,acceleration load,Unnamed: 17,session_id,fixture_id,ts
0,1729429469000,20.10.2024 15:04:29.000,43372,130,74,Vincent Büchner,1262014,1,TSV Hannover-Burgdorf,7.058,...,0.365,NaN,-0.065,0.0,1.287,0.364,NaN,2658,00c08679-4374-11ef-80bd-73cf0bc66b45,2024-10-20 13:04:29+00:00
1,1729429469000,20.10.2024 15:04:29.000,43478,2024,39,Lukas Stutzke,906726,1,TSV Hannover-Burgdorf,8.352,...,0.444,NaN,0.009,0.0,1.608,0.540,NaN,2658,00c08679-4374-11ef-80bd-73cf0bc66b45,2024-10-20 13:04:29+00:00
2,1729429469000,20.10.2024 15:04:29.000,43590,1833,2,Simon Pytlick,1560268,2,SG Flensburg-Handewitt,18.184,...,0.919,2.681,0.267,0.0,3.787,0.735,NaN,2658,00c08679-4374-11ef-80bd-73cf0bc66b45,2024-10-20 13:04:29+00:00


In [19]:
# Cell 9 (revised): Prepare match goal rows with league_id mappings and the Kinexon matched timestamp

# We assume df_me (goals) and df_enriched_matched (with match_eventId + match_time_diff_ms) exist.
# Join league_id for shooter and goalkeeper (for highlighting)
players_core = con.execute("""
    SELECT personId, league_id, nameFullLatin AS person_name, teamName
    FROM players
""").fetchdf()

goals = df_me.copy()
goals = goals.merge(
    players_core.rename(columns={
        "personId": "personId",
        "league_id": "person_league_id",
        "person_name": "personNamePlayers"
    }),
    on="personId", how="left"
)
goals = goals.merge(
    players_core.rename(columns={
        "personId": "goalKeeperId",
        "league_id": "goalkeeper_league_id",
        "person_name": "goalkeeperNamePlayers"
    }),
    on="goalKeeperId", how="left"
)

# Parse times
goals["eventTime"] = pd.to_datetime(goals["eventTime"], utc=True, errors="coerce")
goals["eventTime_ms"] = (goals["eventTime"].astype("int64") // 10**6)

# === NEW: derive the precise Kinexon detected time (ms) that matched this goal ===
# From df_enriched_matched pick, for each (fixture_id, match_eventId), the row with minimal |match_time_diff_ms|
m = (
    df_enriched_matched
      .dropna(subset=["match_eventId", "match_time_diff_ms"])
      .assign(abs_dt=lambda d: d["match_time_diff_ms"].abs())
      .sort_values(["fixture_id", "match_eventId", "abs_dt"])
      .groupby(["fixture_id", "match_eventId"], as_index=False)
      .first()[["fixture_id", "match_eventId", "timestamp_ms", "match_time_diff_ms"]]
      .rename(columns={
          "fixture_id": "fixtureId",
          "match_eventId": "eventId",
          "timestamp_ms": "kinexon_matched_ts_ms"
      })
)

# Merge Kinexon detected time (ms) and delta into goals
goals = goals.merge(m, on=["fixtureId", "eventId"], how="left")
goals["kinexon_matched_ts"] = pd.to_datetime(goals["kinexon_matched_ts_ms"], unit="ms", utc=True, errors="coerce")

print(f"Goals prepared for rendering: {len(goals):,}")
cols_show = [
    "fixtureId","eventId","eventType",
    "eventTime","eventTime_ms",
    "kinexon_matched_ts","kinexon_matched_ts_ms",
    "match_time_diff_ms",
    "person_league_id","goalkeeper_league_id"
]
display(goals[cols_show].head(5))


Goals prepared for rendering: 29,589


,fixtureId,eventId,eventType,eventTime,eventTime_ms,kinexon_matched_ts,kinexon_matched_ts_ms,match_time_diff_ms,person_league_id,goalkeeper_league_id
0,00c08679-4374-11ef-80bd-73cf0bc66b45,09e0f481-8ee4-11ef-baff-6d233f85da92,goal,2024-10-20 13:05:54.160000+00:00,1729429554160,2024-10-20 13:05:51.049000+00:00,1.729430e+12,3111.0,2104590,592178
1,00c08679-4374-11ef-80bd-73cf0bc66b45,2d298c41-8ee4-11ef-baff-6d233f85da92,goal,2024-10-20 13:06:53.356000+00:00,1729429613356,2024-10-20 13:06:27.462000+00:00,1.729430e+12,25894.0,259941,592178
2,00c08679-4374-11ef-80bd-73cf0bc66b45,45066771-8ee4-11ef-baff-6d233f85da92,goal,2024-10-20 13:07:33.431000+00:00,1729429653431,2024-10-20 13:07:31.504000+00:00,1.729430e+12,1927.0,1968457,592178
3,00c08679-4374-11ef-80bd-73cf0bc66b45,5444d6e1-8ee4-11ef-baff-6d233f85da92,goal,2024-10-20 13:07:58.966000+00:00,1729429678966,2024-10-20 13:07:57.454000+00:00,1.729430e+12,1512.0,906726,592178
4,00c08679-4374-11ef-80bd-73cf0bc66b45,67cc1391-8ee4-11ef-baff-6d233f85da92,goal,2024-10-20 13:08:31.729000+00:00,1729429711729,2024-10-20 13:08:29.510000+00:00,1.729430e+12,2219.0,1560268,925204


In [20]:
# Cell 10 (revised): Enhanced renderer with 1.5 s freeze on Kinexon matched timestamp ("ball release")

FREEZE_SECONDS = 1.5  # requested freeze duration
FPS = 20              # keep consistent with earlier cells
FRAME_INTERVAL_MS = int(round(1000 / FPS))
FREEZE_FRAMES = int(round(FREEZE_SECONDS * FPS))

def render_goal_locally(
    df_positions: pd.DataFrame,
    row_goal: pd.Series,
    field_image_path: Path = FIELD_IMAGE,
    out_dir: Path = OUT_DIR,
    fps: int = FPS,
    flash_on_freeze: bool = True,
) -> Path:
    """
    Render a visualization of the goal event based on Kinexon positions and a match goal row.
    Adds a 1.5 s freeze at the detected Kinexon time (ball release).
    """
    img = cv2.imread(str(field_image_path))
    if img is None:
        print(f"⚠️ Could not read field image at: {field_image_path}")
        return None

    height, width = img.shape[:2]
    scale = width / 40.0  # meters-to-pixels assuming 40m width

    name_event = f'event_{row_goal["eventId"]}'
    out_path = out_dir / f"{name_event}.mp4"
    out_path_img = out_dir / f"{name_event}.png"

    writer = cv2.VideoWriter(
        str(out_path),
        cv2.VideoWriter_fourcc(*"mp4v"),
        fps,
        (width, height),
    )

    # Extract IDs for highlights
    shooter_league_id = row_goal.get("person_league_id", None)
    goalkeeper_league_id = row_goal.get("goalkeeper_league_id", None)

    # Window around event (same as before)
    t0 = row_goal["eventTime"] - pd.Timedelta(seconds=POS_PAD_SEC_BEFORE)
    t1 = row_goal["eventTime"] + pd.Timedelta(seconds=POS_PAD_SEC_AFTER)

    # Slice positions to window
    df_scene = df_positions[(df_positions["ts"] >= t0) & (df_positions["ts"] <= t1)].copy()
    if df_scene.empty:
        print(f"⚠️ No positional data in window for eventId={row_goal['eventId']}")
        return None
    df_scene = df_scene.sort_values("ts").copy()

    # Frame sequencing
    df_scene["frame_idx"] = df_scene.groupby("ts").ngroup()

    # Previous positions for ball velocity arrow
    df_scene["prev_x"] = df_scene.groupby(["mapped id"])["x in m"].shift(1)
    df_scene["prev_y"] = df_scene.groupby(["mapped id"])["y in m"].shift(1)

    # Freeze target (Kinexon detected time of the matched event)
    kin_ms = row_goal.get("kinexon_matched_ts_ms", None)
    freeze_done = False  # ensure single freeze

    # Helpers
    def color_for_group(group_id: float):
        if group_id == 3:  return (0, 0, 255), 15   # ball
        if group_id == 2:  return (0, 200, 0), 10
        if group_id == 1:  return (255, 140, 0), 10
        return (200, 200, 200), 8

    def draw_overlay(img_draw, ts):
        # HUD: metadata
        hud_lines = [
            f"Fixture: {row_goal.get('fixtureId','')}",
            f"EventId: {row_goal.get('eventId','')}",
            f"Δt ms (Kinexon↔Goal): {row_goal.get('match_time_diff_ms','N/A')}",
            f"Shooter league_id: {row_goal.get('person_league_id','N/A')}  |  GK league_id: {row_goal.get('goalkeeper_league_id','N/A')}",
            f"Frame time (UTC): {ts.strftime('%Y-%m-%d %H:%M:%S.%f')[:-3]}",
            f"Kinexon match (UTC): {str(row_goal.get('kinexon_matched_ts'))}",
        ]
        y0 = 28
        for line in hud_lines:
            cv2.putText(img_draw, line, (10, y0), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255,255,255), 2, cv2.LINE_AA)
            y0 += 24

    first_png_saved = False

    for ts, group in df_scene.groupby("ts"):
        img_draw = img.copy()

        # Draw actors
        for _, r in group.iterrows():
            gid = r.get("group id", None)
            color, radius = color_for_group(gid)
            if pd.isna(r["x in m"]) or pd.isna(r["y in m"]):
                continue
            x = int(float(r["x in m"]) * scale)
            y = int(float(r["y in m"]) * scale)
            cv2.circle(img_draw, (x, y), radius, color, -1, lineType=cv2.LINE_AA)

            # Labels
            league_id = r.get("league id", "N/A")
            name = r.get("full name", "N/A")
            cv2.putText(img_draw, f"{name}", (x - 12, y - radius - 18),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255, 255, 255), 1, cv2.LINE_AA)
            cv2.putText(img_draw, f"ID:{league_id}", (x - 12, y - radius - 2),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.42, (255, 255, 255), 1, cv2.LINE_AA)

            # Shooter / GK halo
            try:
                if shooter_league_id is not None and int(r.get("league id", -1)) == int(shooter_league_id):
                    cv2.circle(img_draw, (x, y), radius + 6, (255, 255, 0), 2, cv2.LINE_AA)
                if goalkeeper_league_id is not None and int(r.get("league id", -1)) == int(goalkeeper_league_id):
                    cv2.circle(img_draw, (x, y), radius + 6, (0, 255, 255), 2, cv2.LINE_AA)
            except Exception:
                pass

        # Trails (last few frames)
        current_idx = group["frame_idx"].iloc[0]
        TRAIL_FRAMES = 5
        if TRAIL_FRAMES > 0:
            trail_slice = df_scene[
                (df_scene["frame_idx"] <= current_idx) &
                (df_scene["frame_idx"] > current_idx - TRAIL_FRAMES)
            ]
            for _, r in trail_slice.iterrows():
                if pd.isna(r["x in m"]) or pd.isna(r["y in m"]):
                    continue
                x_t = int(float(r["x in m"]) * scale)
                y_t = int(float(r["y in m"]) * scale)
                gid_t = r.get("group id", None)
                color_t, _ = color_for_group(gid_t)
                cv2.circle(img_draw, (x_t, y_t), 3, color_t, -1, cv2.LINE_AA)

        # Ball velocity arrow
        for _, br in group[group["group id"] == 3].iterrows():
            if not (pd.isna(br["prev_x"]) or pd.isna(br["prev_y"])):
                x0, y0 = int(br["prev_x"] * scale), int(br["prev_y"] * scale)
                x1, y1 = int(br["x in m"] * scale), int(br["y in m"] * scale)
                cv2.arrowedLine(img_draw, (x0, y0), (x1, y1), (50,50,255), 2, tipLength=0.3)

        # Overlay
        draw_overlay(img_draw, ts)

        # Determine whether to freeze on this frame:
        # Freeze when the current frame time is within half a frame of kinexon matched ms.
        if not freeze_done and kin_ms is not None:
            ts_ms = int(ts.value // 10**6)
            if abs(ts_ms - int(kin_ms)) <= (FRAME_INTERVAL_MS // 2):
                # Emphasize the exact moment
                if flash_on_freeze:
                    cv2.rectangle(img_draw, (0,0), (width-1,height-1), (0,255,255), 6, cv2.LINE_AA)
                    cv2.putText(img_draw, "BALL RELEASE", (width//2 - 140, 80),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0,255,255), 2, cv2.LINE_AA)

                # Save PNG preview at the exact freeze frame
                if not first_png_saved:
                    cv2.imwrite(str(out_path_img), img_draw)
                    first_png_saved = True

                # Write this same frame multiple times to create a freeze (1.5 s)
                for _ in range(FREEZE_FRAMES):
                    writer.write(cv2.resize(img_draw, (width, height)))
                freeze_done = True  # only once
                # Also show during freeze if display is available
                if has_display():
                    for _ in range(FREEZE_FRAMES):
                        cv2.imshow("Goal Render", img_draw)
                        if cv2.waitKey(FRAME_INTERVAL_MS) & 0xFF == ord('q'):
                            break

                # Continue to next time step without additional write below
                continue

        # Save first PNG if not already
        if not first_png_saved:
            cv2.imwrite(str(out_path_img), img_draw)
            first_png_saved = True

        # Show (if display available)
        if has_display():
            cv2.imshow("Goal Render", img_draw)
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break

        # Write the normal frame
        writer.write(cv2.resize(img_draw, (width, height)))

    writer.release()
    if has_display():
        cv2.destroyAllWindows()

    print(f"🎬 Saved: {out_path.name}  | 🖼️ Preview: {out_path_img.name}")
    return out_path


In [23]:
# Cell 11: Run renders for a small sample (e.g., first 5 goals per fixture)

# Optionally restrict to a specific fixtureId to speed up
# goals = goals[goals["fixtureId"] == "<your-fixture-id>"].copy()

# Preload positions per fixture to minimize scanning:
# We map fixture -> session_id using kinexon_positions (1 fixture usually 1 session)
fx_sessions = con.execute("""
    SELECT DISTINCT fixture_id, ANY_VALUE(session_id) AS session_id
    FROM kinexon_positions
    WHERE fixture_id IS NOT NULL
    GROUP BY fixture_id
""").fetchdf()

fx_to_session = dict(zip(fx_sessions["fixture_id"], fx_sessions["session_id"]))

render_count = 0
RENDER_LIMIT = 10  # change as needed

for _, row in goals.iterrows():
    fixture_id = row["fixtureId"]
    session_id = fx_to_session.get(fixture_id, None)
    if session_id is None:
        print(f"⚠️ No session_id for fixture {fixture_id}, skipping")
        continue

    # Narrow positions to this fixture/session for speed
    df_pos_fx = df_positions_all[df_positions_all["session_id"] == session_id].copy()
    if df_pos_fx.empty:
        print(f"⚠️ No positions for session {session_id}, skipping")
        continue

    # Render
    render_goal_locally(df_pos_fx, row)

    render_count += 1
    if render_count >= RENDER_LIMIT:
        break

print(f"Done. Rendered {render_count} event(s).")


🎬 Saved: event_09e0f481-8ee4-11ef-baff-6d233f85da92.mp4  | 🖼️ Preview: event_09e0f481-8ee4-11ef-baff-6d233f85da92.png
🎬 Saved: event_2d298c41-8ee4-11ef-baff-6d233f85da92.mp4  | 🖼️ Preview: event_2d298c41-8ee4-11ef-baff-6d233f85da92.png
🎬 Saved: event_45066771-8ee4-11ef-baff-6d233f85da92.mp4  | 🖼️ Preview: event_45066771-8ee4-11ef-baff-6d233f85da92.png
🎬 Saved: event_5444d6e1-8ee4-11ef-baff-6d233f85da92.mp4  | 🖼️ Preview: event_5444d6e1-8ee4-11ef-baff-6d233f85da92.png
🎬 Saved: event_67cc1391-8ee4-11ef-baff-6d233f85da92.mp4  | 🖼️ Preview: event_67cc1391-8ee4-11ef-baff-6d233f85da92.png
🎬 Saved: event_6cf6b550-8ee4-11ef-baff-6d233f85da92.mp4  | 🖼️ Preview: event_6cf6b550-8ee4-11ef-baff-6d233f85da92.png
🎬 Saved: event_733d2750-8ee4-11ef-baff-6d233f85da92.mp4  | 🖼️ Preview: event_733d2750-8ee4-11ef-baff-6d233f85da92.png
🎬 Saved: event_bd5e8271-8ee4-11ef-914a-5f491cb312bd.mp4  | 🖼️ Preview: event_bd5e8271-8ee4-11ef-914a-5f491cb312bd.png
🎬 Saved: event_e21e50e1-8ee4-11ef-914a-5f491cb312bd.mp4 

ValueError: cannot convert float NaN to integer

In [ ]:
# Cell 12 (optional): Render a specific goal by eventId

EVENT_ID_TO_RENDER = None  # e.g., "b7d8cd90-8ed2-11ef-b16d-cda25f1166cf"
if EVENT_ID_TO_RENDER:
    row = goals[goals["eventId"] == EVENT_ID_TO_RENDER].head(1)
    if not row.empty:
        row = row.iloc[0]
        session_id = fx_to_session.get(row["fixtureId"], None)
        if session_id is not None:
            df_pos_fx = df_positions_all[df_positions_all["session_id"] == session_id].copy()
            render_goal_locally(df_pos_fx, row)
        else:
            print("No session for that fixture.")
    else:
        print("EventId not found in goal list.")
